In [ ]:
import numpy as np
import pandas as pd

d5s = pd.read_csv("../outputs/aug_all_scored.csv")
base = (d5s[d5s["operator"] == "baseline"][["clip_id", "corpus", "log_parity_time", "z"]]
        .dropna().reset_index(drop=True))
base["allocation"] = 10 * np.clip(base["log_parity_time"] / np.log(4), -1, 1)
print("clips in pool:", len(base), "| AMI:", int((base["corpus"] == "AMI").sum()),
      "| MELD:", int((base["corpus"] == "MELD").sum()))

N_SHARDS, SIZE, SEED = 200, 30, 2026
rng = np.random.default_rng(SEED)
rows = []
for s in range(N_SHARDS):
    for i in rng.choice(len(base), size=SIZE, replace=False):
        rows.append(("shard_%03d" % s, base.loc[i, "clip_id"]))
c2s = pd.DataFrame(rows, columns=["shard_id", "clip_id"])
c2s.to_csv("../outputs/shared/clip_to_shard.csv", index=False)

m = c2s.merge(base, on="clip_id")
sh = m.groupby("shard_id").agg(n_clips=("clip_id", "size"),
                               n_AMI=("corpus", lambda x: int((x == "AMI").sum())),
                               n_MELD=("corpus", lambda x: int((x == "MELD").sum())),
                               allocation=("allocation", "mean"),
                               structure_z=("z", "mean")).reset_index()
sh.to_csv("../outputs/shard_table.csv", index=False)

use = c2s["clip_id"].value_counts()
print("shards:", sh.shape[0], "| clips per shard:", int(sh["n_clips"].min()), "-", int(sh["n_clips"].max()))
print("times each clip is used: min %d, median %d, max %d" % (use.min(), use.median(), use.max()))
print("shard allocation: mean %.2f, SD %.2f" % (sh["allocation"].mean(), sh["allocation"].std()))
print("shard structure z: mean %.2f, SD %.2f, 95%% band +/- %.2f" %
      (sh["structure_z"].mean(), sh["structure_z"].std(), sh["structure_z"].abs().quantile(0.95)))
print(sh.head().round(2).to_string(index=False))

In [ ]:
import json
import numpy as np

spec = json.load(open("../outputs/m2_model_v0.1.json"))
f = spec["features"]
mu = np.array(spec["scaler_mean"])
sc = np.array(spec["scaler_scale"])
w = np.array(spec["coef"])
b = float(spec["intercept"])

w_raw = w / sc
b_raw = b - float((w * mu / sc).sum())

print("features:", f)
print("standardised weights (compare importance):", dict(zip(f, np.round(w, 3))))
terms = ["%+.3f x %s" % (wi, fi) for wi, fi in zip(w_raw, f)]
print("raw = " + " ".join(terms) + " %+.3f" % b_raw)
print("Structure z = (raw - %.3f) / %.3f" % (spec["null_mean"], spec["null_sd"]))
print("ridge alpha:", spec.get("alpha"))

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

d = pd.read_csv("../outputs/aug_all_scored.csv")
cols = ["struct_asym", "interruption_asym", "position_gap", "pos_sd_ratio"]
d = d.dropna(subset=cols + ["label"]).reset_index(drop=True)
print("rows:", len(d), "| clips:", d["clip_id"].nunique())

print("Correlation among features (unmodified clips):")
print(d[d["operator"] == "baseline"][cols].corr().round(2).to_string())
print("Correlation among features (all augmented rows):")
print(d[cols].corr().round(2).to_string())

Z = (d[cols] - d[cols].mean()) / d[cols].std()
X = sm.add_constant(Z)
y = d["label"]

ols = sm.OLS(y, X).fit()
rob = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["clip_id"])[0]})
print("R2: %.3f" % ols.rsquared)
print("Coefficients with clip-clustered standard errors:")
print(rob.summary().tables[1])

print("VIF (below 5 is fine):", {c: round(float(variance_inflation_factor(X.values, i + 1)), 2)
                                  for i, c in enumerate(cols)})
bp = het_breuschpagan(ols.resid, X)
print("Breusch-Pagan p-value (small = non-constant residual variance): %.4g" % bp[1])
jb = jarque_bera(ols.resid)
print("Jarque-Bera p-value: %.4g | skew %.2f | kurtosis %.2f" % (jb[1], jb[2], jb[3]))
print("Mean residual: %.4f" % ols.resid.mean())

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, os.path.abspath("../src"))
importlib.invalidate_caches()
import m2_temporal as m2

allv = pd.read_csv("../outputs/shared/intervals_all.csv")
scored = pd.read_csv("../outputs/aug_all_scored.csv")
sb = scored[scored["operator"] == "baseline"].copy()
keep_clips = set(sb["clip_id"])

df = allv[allv["clip_id"].isin(keep_clips) & allv["gender"].isin(["F", "M"])].copy()
df = df.rename(columns={"clip_id": "meeting", "person_id": "speaker"})
df["role"] = np.nan
df["n_words"] = df["n_words"].astype(float)
bad = (df["corpus"] == "MELD") & (df["dur"] > 8) & (df["n_words"] / df["dur"] < 0.5)
df_clean = df[~bad].copy()
print("intervals:", len(df), "| removed in cleaned variant:", int(bad.sum()))

scorer = m2.M2Scorer.load("../outputs/m2_model_v0.1.json")

def run(d, bc, gap):
    t = m2.build_turns(m2.mark_backchannels(d, bc), gap=gap)
    rows = []
    for cid, tc in t.groupby("meeting"):
        r = m2.score_clip(tc, scorer, cid)
        r["corpus"] = cid.split(":")[0]
        rows.append(r)
    return pd.DataFrame(rows).set_index("clip_id")

variants = {"base (backchannel<=2, gap 1.0)": (df, 2, 1.0),
            "backchannel<=1": (df, 1, 1.0),
            "backchannel<=3": (df, 3, 1.0),
            "gap 0.5": (df, 2, 0.5),
            "gap 2.0": (df, 2, 2.0),
            "MELD cleaned": (df_clean, 2, 1.0)}
res = {k: run(*v) for k, v in variants.items()}
base = res["base (backchannel<=2, gap 1.0)"]

chk = sb.set_index("clip_id")["log_parity_time"]
a_old = 10 * np.clip(chk / np.log(4), -1, 1)
common = base.index.intersection(a_old.index)
print("sanity: module base vs earlier pipeline, allocation | clips:", len(common),
      "| max abs diff: %.3f" % float((base.loc[common, "allocation_score"] - a_old.loc[common]).abs().max()),
      "| Spearman: %.3f" % spearmanr(base.loc[common, "allocation_score"], a_old.loc[common])[0])

rows = []
for k, r in res.items():
    r = r.reindex(base.index)
    ok = base["status_structure"].eq("valid") & r["status_structure"].eq("valid")
    okm = ok & (base["corpus"] == "MELD")
    rows.append({"variant": k,
                 "clips with structure score": int(r["status_structure"].eq("valid").sum()),
                 "Spearman allocation vs base": round(spearmanr(base.loc[ok, "allocation_score"], r.loc[ok, "allocation_score"])[0], 3),
                 "Spearman structure vs base": round(spearmanr(base.loc[ok, "structure_z"], r.loc[ok, "structure_z"])[0], 3),
                 "mean allocation": round(float(r["allocation_score"].mean()), 2),
                 "flagged |z|>1.96 (%)": round(100 * float((r.loc[ok, "structure_z"].abs() > 1.96).mean()), 1),
                 "MELD flagged (%)": round(100 * float((r.loc[okm, "structure_z"].abs() > 1.96).mean()), 1)})
print(pd.DataFrame(rows).to_string(index=False))